In [87]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix
import sklearn.metrics.pairwise as dist
import pickle

Chargement des données

In [2]:
ratings = pd.read_csv('ratings_cleaned.csv')
movies = pd.read_csv('movies_cleaned.csv')

MOVIES

In [3]:
display(movies.head(2))
movies.shape

,movieId,title,genre_Action,genre_Adventure,genre_Animation,genre_Children,genre_Comedy,genre_Crime,genre_Documentary,genre_Drama,...,genre_Film-Noir,genre_Horror,genre_IMAX,genre_Musical,genre_Mystery,genre_Romance,genre_Sci-Fi,genre_Thriller,genre_War,genre_Western
0,1,Toy Story (1995),0,1,1,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),0,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


(27278, 21)

RATINGS

In [4]:
display(ratings.head(2))
ratings.shape

,userId,movieId,rating
0,1,2,3.5
1,1,29,3.5


(20000263, 3)

**********************************************************************

Filtrage collaboratif : on exploite exclusivement les interactions passées dans les utilisateurs et les films, en regroupant et identifiant des groupes d'utilisateurs dont les interactions sont similaires

**********************************************************************

1. Approche mémoire : elle se base sur la corrélation entre les comportements "passés" des utilisateurs

In [5]:
# on crée un dataframe df qui va contenir ratings mais avec les titres de film en plus des identifiants
# On ne s'intéresse qu'aux notes >= 3, car en dessous les notes sont moins pertinentes
df = ratings[ratings['rating'] >= 3]

n_users = df['userId'].nunique()
n_movies = df['movieId'].nunique()
print("Nombre d'utilisateurs : ", n_users)
print("Nombre de films : ", n_movies)
print(f"La matrice dense aura pour taille : {n_users*n_movies:,}")

Nombre d'utilisateurs :  138445
Nombre de films :  24800
La matrice dense aura pour taille : 3,433,436,000


Nous avons 2 types d'approche mémoire : User-based filtering (similarité entre les utilisateurs) et Item-based filtering (similarité entre les films). La similarité sur les utilisateurs va construire une matrice de dimension 138445*138445 soit 19 167 018 025 cellules. Cette méthode n'est pas envisageable sur un tel nombre d'utilisateurs. Il faudrait réduire considérablement le nombre d'utilisateurs ce qui biaiserait énormément notre jeu de données. En alternative on pourrait mettre en place la méthode des k plus proches voisins. Nous n'appliquerons donc que la méthode basée sur la similarité entre les films

Item-based filtering : on ne mesure pas la corrélation entre des utilisateurs mais entre le contenu (films). Le but est de trouver des films similaire aux films que l'utilisateur cible a beaucoup aimés. La matrice de similarité aura pour dimension 24800 * 24800, soit 615 040 000 cellules. Le calcul est envisageable, mais il va devoir passer par une matrice de notation de dimension 24800 * 138445, ce qui est assez coûteux.

In [6]:
# matrice mat_ratings de notation associée au dataframe en prenant en index les movieId et en colonnes les userId
mat_ratings = df.pivot(index='movieId', columns='userId', values='rating')
display(mat_ratings.info())
print("Dimension de la matrice de notation : ", mat_ratings.shape)
mat_ratings.head(2)

/var/folders/43/j4zwb6d13hd0m383lzvlyljr0000gn/T/ipykernel_51251/381608062.py:2: PerformanceWarning: The following operation may generate 3433436000 cells in the resulting pandas object.
  mat_ratings = df.pivot(index='movieId', columns='userId', values='rating')


<class 'pandas.DataFrame'>
Index: 24800 entries, 1 to 131262
Columns: 138445 entries, 1 to 138493
dtypes: float64(138445)
memory usage: 25.6 GB


None

Dimension de la matrice de notation :  (24800, 138445)


userId,1,2,3,4,5,6,7,8,9,10,...,138484,138485,138486,138487,138488,138489,138490,138491,138492,138493
movieId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,4.0,NaN,NaN,5.0,NaN,4.0,NaN,4.0,...,NaN,NaN,5.0,NaN,3.0,NaN,NaN,NaN,NaN,3.5
2,3.5,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN,NaN,...,3.0,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN,4.0


La matrice de notation est assez dense et fait 25,6GB ce qui est assez conséquent pour la mémoire de nos ordinateurs

Alternative : En construisant la matrice creuse **directement à partir des triplets** (movieId, userId, rating), sans jamais matérialiser de tableau dense, on peut optimiser le temps de calcul 

In [7]:
# encodage des movieId et userId en indices entiers consécutifs (0..n-1), nécessaire pour
# construire directement la matrice creuse à partir des triplets
movie_cat = df['movieId'].astype('category')
user_cat = df['userId'].astype('category')

moviesId = movie_cat.cat.categories.tolist()
userIds = user_cat.cat.categories.tolist()

# construction de la matrice creuse (films x users) sans passer par une matrice dense
sparse_ratings = csr_matrix(
    (df['rating'].values, (movie_cat.cat.codes.values, user_cat.cat.codes.values)),
    shape=(len(moviesId), len(userIds))
)

print("Dimension de la matrice creuse :", sparse_ratings.shape)

Dimension de la matrice creuse : (24800, 138445)


In [8]:
# Utilisation de la fonction 'cosine_similarity' du module 'dist' pour calculer la similarité cosinus entre les utilisateurs.
item_similarity = dist.cosine_similarity(sparse_ratings)
# Création d'un DataFrame pandas à partir de la matrice de similarité entre items.
# Les index et les colonnes du DataFrame sont les identifiants des films.
item_similarity = pd.DataFrame(item_similarity, index=moviesId, columns=moviesId)
# on nomme l'index pour que reset_index() produise une colonne 'movieId' (nécessaire pour les merges)
item_similarity.index.name = 'movieId'

In [9]:
# fonction qui prend en entrée un movieId et renvoie les 10 films les plus similaires à ce film
def get_similar_movies(movieId, item_similarity, top_n=10):
    # Vérifie si le movieId est présent dans la matrice de similarité
    if movieId not in item_similarity.index:
        return pd.DataFrame()  # Retourne un DataFrame vide si le movieId n'est pas trouvé

    # Récupère les similarités pour le film donné
    similar_scores = item_similarity[movieId]

    # Trie les films par similarité décroissante et sélectionne les top_n films similaires
    top_similar_movies = similar_scores.sort_values(ascending=False).head(top_n + 1)  # +1 pour exclure le film lui-même

    # Exclut le film lui-même de la liste des films similaires
    top_similar_movies = top_similar_movies[top_similar_movies.index != movieId]

    return top_similar_movies

In [77]:
#recherche de film à tester
movies[movies['title'].str.contains('scarface', case=False)][['movieId', 'title']]

,movieId,title
4168,4262,Scarface (1983)
8379,25788,Scarface (1932)


In [78]:
# recherche l'utlisateur qui a regardé le movieId
mat_ratings.loc[4262].dropna().sort_values(ascending=False)

userId
59527     5.0
23447     5.0
100623    5.0
63555     5.0
119017    5.0
         ... 
91972     3.0
116454    3.0
56574     3.0
126524    3.0
32928     3.0
Name: 4262, Length: 9663, dtype: float64

In [79]:
movieId = 4262
similar_movies = get_similar_movies(movieId, item_similarity, top_n=10)
# Ajouter les titres des films similaires
similar_movies = similar_movies.reset_index()
similar_movies = similar_movies.merge(movies[['movieId', 'title']], left_on='movieId', right_on='movieId', how='left')
print(similar_movies)

   movieId      4262                                     title
0     1089  0.434391                     Reservoir Dogs (1992)
1     4011  0.428644                             Snatch (2000)
2     1222  0.427236                  Full Metal Jacket (1987)
3     1213  0.419698                         Goodfellas (1990)
4     6874  0.418745                  Kill Bill: Vol. 1 (2003)
5     2329  0.410661                 American History X (1998)
6     7438  0.404258                  Kill Bill: Vol. 2 (2004)
7    32587  0.401550                           Sin City (2005)
8     2959  0.397994                         Fight Club (1999)
9     2542  0.389539  Lock, Stock & Two Smoking Barrels (1998)


In [20]:
def pred_item(mat_ratings, item_similarity, k, user_id):

    # Sélectionner dans mat_ratings les films qui n'ont pas été encore vus par le user
    # (mat_ratings a movieId en index et userId en colonnes, donc on sélectionne la colonne user_id)
    to_predict = mat_ratings[user_id]
    to_predict = to_predict[to_predict.isna()]
    print(f"Nombre de films à prédire pour l'utilisateur {user_id} : {len(to_predict)}")

    # Itérer sur tous ces films 
    for i in to_predict.index:

        #Trouver les k films les plus similaires en excluant le film lui-même
        similar_items = item_similarity.loc[i].sort_values(ascending=False)[1:k+1]

        # Calcul de la norme du vecteur similar_items
        norm = np.sum(np.abs(similar_items))

        # Récupérer les notes données par l'utilisateur aux k plus proches voisins
        ratings = mat_ratings.loc[similar_items.index, user_id].fillna(0)


        # Calculer le produit scalaire entre ratings et similar_items
        scalar_prod = np.dot(ratings,similar_items)
        
        #Calculer la note prédite pour le film i
        pred = scalar_prod / norm

        # Remplacer par la prédiction
        to_predict[i] = pred


    return to_predict

In [80]:
# Top notations de l'utilisateur userId
#userId = 31
#userId = 1
#userId = 521
#userId = 129058
#userId = 37464
#userId = 138474
userId = 59527
user_preferences = df[(df['userId']==userId) & (df['rating']>=4)]
user_preferences = user_preferences.sort_values('rating', ascending=False).drop_duplicates().head(10)
user_preferences.merge(movies[['movieId', 'title']], left_on='movieId', right_on='movieId', how='left')

,userId,movieId,rating,title
0,59527,1,5.0,Toy Story (1995)
1,59527,1089,5.0,Reservoir Dogs (1992)
2,59527,1617,5.0,L.A. Confidential (1997)
3,59527,7153,5.0,"Lord of the Rings: The Return of the King, The..."
4,59527,6377,5.0,Finding Nemo (2003)
5,59527,48780,5.0,"Prestige, The (2006)"
6,59527,1225,5.0,Amadeus (1984)
7,59527,6,5.0,Heat (1995)
8,59527,1221,5.0,"Godfather: Part II, The (1974)"
9,59527,1219,5.0,Psycho (1960)


In [81]:
reco_item = pred_item(mat_ratings, item_similarity, 20, userId).sort_values(ascending=False).head(20)

# ajouter les titres des films recommandés
reco_item = reco_item.reset_index()
reco_item = reco_item.merge(movies[['movieId', 'title']], left_on='movieId', right_on='movieId', how='left')

print(reco_item)

Nombre de films à prédire pour l'utilisateur 59527 : 24630
    movieId     59527                                        title
0       111  3.522999                           Taxi Driver (1976)
1      1208  3.274271                        Apocalypse Now (1979)
2      1090  3.260977                               Platoon (1986)
3      2329  3.237019                    American History X (1998)
4        16  3.203438                                Casino (1995)
5      2324  3.086869   Life Is Beautiful (La Vita è bella) (1997)
6      1222  2.983022                     Full Metal Jacket (1987)
7      1207  2.952558                 To Kill a Mockingbird (1962)
8      1249  2.891755             Femme Nikita, La (Nikita) (1990)
9       431  2.812723                         Carlito's Way (1993)
10     5782  2.795755  Professional, The (Le professionnel) (1981)
11      923  2.732210                          Citizen Kane (1941)
12    30749  2.611946                          Hotel Rwanda (2004)
13 

In [83]:
# utilisateurs utilisés pour la demo streamlit
user_ids_demo = [1, 31, 521, 129058, 37464, 138474, 59527]

resultats_demo = {}
for uid in user_ids_demo:
    predictions = pred_item(mat_ratings, item_similarity, 20, uid)
    df_pred = predictions.reset_index()
    df_pred.columns = ['movieId', 'note_predite']
    df_pred = df_pred.merge(movies[['movieId', 'title']], on='movieId', how='left')
    df_pred = df_pred[['title', 'note_predite']].head(30)
    df_pred['note_predite'] = df_pred['note_predite'].round(2)
    
    resultats_demo[uid] = df_pred

Nombre de films à prédire pour l'utilisateur 1 : 24625
Nombre de films à prédire pour l'utilisateur 31 : 24620
Nombre de films à prédire pour l'utilisateur 521 : 24664
Nombre de films à prédire pour l'utilisateur 129058 : 24334
Nombre de films à prédire pour l'utilisateur 37464 : 24527
Nombre de films à prédire pour l'utilisateur 138474 : 24218
Nombre de films à prédire pour l'utilisateur 59527 : 24630


In [86]:
# resultats_demo est un doctionnaire dont la clé de claque élement est le userId. Chaque élément du dictionnaire est un dataframe
resultats_demo[1].head()

,title,note_predite
0,Toy Story (1995),1.39
1,Grumpier Old Men (1995),0.13
2,Waiting to Exhale (1995),0.00
3,Father of the Bride Part II (1995),0.00
4,Heat (1995),1.65


In [88]:
#Export du dictionnaire
with open('demo_recommandations.pkl', 'wb') as f:
    pickle.dump(resultats_demo, f)